# 🚀 YOLOv8 Multi-Task Training Notebook v2
## Object Detection, Instance Segmentation & Classification

---

**Supported Tasks:**
- `detect` - Object Detection (bounding boxes)
- `segment` - Instance Segmentation (boxes + pixel masks)  
- `classify` - Image Classification (whole image labels)

**Steps:**
1. Enable GPU runtime
2. Install dependencies
3. Select task type
4. Download dataset from Roboflow
5. Configure & train model
6. Evaluate & download results

## 1️⃣ Enable GPU Runtime
**Go to: Runtime → Change runtime type → T4 GPU**

In [ ]:
# Check GPU
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → GPU")

## 2️⃣ Install Dependencies

In [ ]:
# Force upgrade to get YOLOv11 support
!pip install -U ultralytics roboflow
print("✅ Dependencies installed!")

## 3️⃣ Select Training Task

In [ ]:
print("="*60)
print("🎯 SELECT TRAINING TASK")
print("="*60)
print("""
Available Tasks:
  detect   - Object Detection (bounding boxes)
  segment  - Instance Segmentation (boxes + pixel masks)
  classify - Image Classification (whole image labels)
""")

task_type = input("Enter task (detect/segment/classify) [default: detect]: ").strip().lower() or "detect"

# Validate
if task_type not in ["detect", "segment", "classify"]:
    print(f"⚠️ Invalid '{task_type}'. Using 'detect'")
    task_type = "detect"

# Set format and model suffix
task_config = {
    "detect": {"format": "yolov8", "suffix": "", "desc": "Object Detection"},
    "segment": {"format": "yolov8-seg", "suffix": "-seg", "desc": "Instance Segmentation"},
    "classify": {"format": "folder", "suffix": "-cls", "desc": "Classification"}
}

roboflow_format = task_config[task_type]["format"]
model_suffix = task_config[task_type]["suffix"]
task_desc = task_config[task_type]["desc"]

print(f"\n✅ Task: {task_desc}")
print(f"   Format: {roboflow_format}")


## 4️⃣ Download Dataset from Roboflow

In [ ]:
# ============================================
# 📥 DOWNLOAD DATASET FROM ROBOFLOW
# ============================================
# PASTE YOUR ROBOFLOW CODE BELOW (from "Show download code"):
# ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1)
dataset = version.download(roboflow_format) # Use the format variable (yolov8 or folder)

# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑
# ============================================

# ✅ FIX: Handle Classification paths correctly
if task_type == "classify":
    # Classification uses the FOLDER path itself, not a yaml file
    data_path = dataset.location
    print(f"✅ Classification data path: {data_path}")
else:
    # Detect/Segment use the YAML file
    data_path = f"{dataset.location}/data.yaml"
    print(f"✅ Config file: {data_path}")

print(f"📂 Downloaded to: {dataset.location}")

# Check dataset structure
import os
if os.path.exists(data_path) and task_type != "classify":
    with open(data_path, 'r') as f:
        print(f"\n📋 data.yaml contents:\n{f.read()}")
elif task_type == "classify":
    print(f"\n📋 Folders found: {os.listdir(data_path)}")


## 5️⃣ Configure Training

In [ ]:
# ===========================
# SELECT YOLO VERSION
# ===========================
print("🤖 YOLO Version Options: 8 (stable) or 11 (latest)")
yolo_version = input("YOLO version [default: 11]: ").strip() or "11"

# Validate
if yolo_version not in ["8", "11"]:
    print(f"⚠️ Invalid '{yolo_version}'. Using '11'")
    yolo_version = "11"

# Build base name
if yolo_version == "8":
    yolo_base = "yolov8"
else:
    yolo_base = "yolo11"

# ===========================
# MODEL SIZE & TRAINING CONFIG
# ===========================
print("\n📐 Model Size Options: n(nano) s(small) m(medium) l(large) x(xlarge)")
model_size = input("Model size [default: m]: ").strip().lower() or "m"
model_name = f"{yolo_base}{model_size}{model_suffix}.pt"

print("\n📊 Training Parameters (press Enter for defaults):")
EPOCHS = int(input("Epochs [100]: ").strip() or "100")
BATCH = int(input("Batch size [16]: ").strip() or "16")
IMGSZ = int(input("Image size [640]: ").strip() or "640")
PATIENCE = int(input("Early stopping patience [50]: ").strip() or "50")

print(f"\n✅ Configuration:")
print(f"   Task: {task_desc}")
print(f"   Model: {model_name}")
print(f"   Epochs: {EPOCHS}, Batch: {BATCH}, Size: {IMGSZ}")

## 6️⃣ Train Model

In [ ]:
from ultralytics import YOLO
import torch

print(f"🚀 Loading {model_name}...")
model = YOLO(model_name)
print(f"   Parameters: {sum(p.numel() for p in model.model.parameters())/1e6:.1f}M")

# Training arguments
train_args = {
    "data": data_path,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "patience": PATIENCE,
    "device": 0 if torch.cuda.is_available() else "cpu",
    "project": f"runs/{task_type}",
    "name": "train",
    "exist_ok": True,
    "pretrained": True,
    "amp": True,
    "plots": True,
    "seed": 42
}

print(f"\n🎬 Starting training...")
print(f"   Saving to: runs/{task_type}/train/")
print("="*60)

results = model.train(**train_args)

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)

## 7️⃣ Evaluate Model

In [ ]:
# ------------------------------------------------------------------------------
# [Section 7] Evaluate Model (Robust Path Finding)
# ------------------------------------------------------------------------------

import os
import glob
from ultralytics import YOLO
from IPython.display import Image, display

# 1. Find the latest 'train' folder dynamically
# This handles the weird "runs/segment/runs/segment/train" nesting bug
search_pattern = f"/content/runs/{task_type}/**/train*/weights/best.pt"
found_models = sorted(glob.glob(search_pattern, recursive=True))

if not found_models:
    # Fallback
    best_path = f"runs/{task_type}/train/weights/best.pt"
else:
    # Use the most recent one found
    best_path = found_models[-1]

print(f"📂 Best model found at: {best_path}")

# 2. Check if file actually exists before loading
if os.path.exists(best_path):
    best_model = YOLO(best_path)

    # 3. Validate
    print("\n🧪 Running validation...")
    val_results = best_model.val(data=data_path)

    # 4. Show metrics
    print("\n📊 Metrics:")
    if task_type == "classify":
        print(f"   Top-1 Accuracy: {val_results.top1:.4f}")
        print(f"   Top-5 Accuracy: {val_results.top5:.4f}")
    else:
        print(f"   Box mAP50: {val_results.box.map50:.4f}")
        print(f"   Box mAP50-95: {val_results.box.map:.4f}")
        if task_type == "segment":
            print(f"   Mask mAP50: {val_results.seg.map50:.4f}")
            print(f"   Mask mAP50-95: {val_results.seg.map:.4f}")

    # 5. Find and show results image (in the same 'train' parent folder)
    train_dir = os.path.dirname(os.path.dirname(best_path)) # Go up two levels
    results_img = os.path.join(train_dir, "results.png")
    
    if os.path.exists(results_img):
        print(f"\n📈 Training Curves ({results_img}):")
        display(Image(filename=results_img))
    else:
        print(f"\n⚠️ Could not find results.png at {results_img}")

else:
    print(f"❌ Error: Model file not found at {best_path}")
    print("Please check the 'runs' folder in the file browser on the left.")


## 8️⃣ Test Predictions

In [ ]:
import cv2
import matplotlib.pyplot as plt
from glob import glob

# Get validation images
if task_type == "classify":
    val_path = f"{dataset.location}/valid"
    test_images = []
    for cls in os.listdir(val_path)[:2]:
        cls_imgs = glob(f"{val_path}/{cls}/*")[:3]
        test_images.extend(cls_imgs)
else:
    test_images = glob(f"{dataset.location}/valid/images/*")[:6]

print(f"🖼️ Testing on {len(test_images)} images...")

# Run predictions
predictions = best_model.predict(test_images, conf=0.25, save=True, project=f"runs/{task_type}", name="predict", exist_ok=True)

# Display
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, pred in zip(axes.flatten(), predictions):
    img = pred.plot()
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9️⃣ Download Model

In [ ]:
# ------------------------------------------------------------------------------
# [Section 9] Download Model (Robust Packaging)
# ------------------------------------------------------------------------------

import shutil
from google.colab import files
import os
import datetime
import glob

# Safe recovery of best_path if variables were cleared
if 'best_path' not in locals() or not os.path.exists(best_path):
    search_pattern = f"/content/runs/{task_type}/**/train*/weights/best.pt"
    found = sorted(glob.glob(search_pattern, recursive=True))
    best_path = found[-1] if found else None

if best_path and os.path.exists(best_path):
    # Derive paths
    weights_dir = os.path.dirname(best_path)
    train_dir = os.path.dirname(weights_dir)
    last_path = os.path.join(weights_dir, "last.pt")
    results_img = os.path.join(train_dir, "results.png")
    confusion_matrix = os.path.join(train_dir, "confusion_matrix.png")

    # Create package
    pkg = "model_package"
    if os.path.exists(pkg):
        shutil.rmtree(pkg)
    os.makedirs(pkg, exist_ok=True)

    print(f"📦 Packaging model from: {train_dir}")
    shutil.copy(best_path, f"{pkg}/best.pt")
    
    if os.path.exists(last_path):
        shutil.copy(last_path, f"{pkg}/last.pt")
    
    if os.path.exists(results_img):
        shutil.copy(results_img, f"{pkg}/results.png")
        
    if os.path.exists(confusion_matrix):
        shutil.copy(confusion_matrix, f"{pkg}/confusion_matrix.png")

    # README
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    with open(f"{pkg}/README.txt", "w") as f:
        f.write(f"DAAN Project - YOLOv8 {task_desc} Model\n")
        f.write(f"Date: {timestamp}\n")
        f.write(f"Model Type: {task_type}\n")
        f.write(f"Base Model: {model_name}\n")
        f.write(f"Epochs: {EPOCHS}\n")
        f.write(f"\nUsage:\n")
        f.write(f"from ultralytics import YOLO\n")
        f.write(f"model = YOLO('best.pt')\n")
        if task_type == "classify":
             f.write(f"results = model.predict('source.jpg')\n")
             f.write(f"print(results[0].probs.top1conf, results[0].names[results[0].probs.top1])\n")
        else:
             f.write(f"results = model.predict('source.mp4', save=True)\n")

    # Zip and download
    zip_name = f"trained_{task_type}_model"
    shutil.make_archive(zip_name, "zip", pkg)
    print(f"📥 Downloading {zip_name}.zip...")
    files.download(f"{zip_name}.zip")
    print("\n🎉 Done!")
else:
    print("❌ Error: Could not verify 'best.pt' path. Check Section 7.")
